# Ninas and pangrams

Two things a setter can ask for on top of the style. Both constrain the fill
rather than the grid, and they work in opposite ways: a nina fixes letters
before the search begins, while a pangram leaves every letter free and changes
which words the search prefers.

1. What a nina is, and why it cannot be a fixed set of cells
2. Placing one, and checking it is actually hidden
3. How fixed letters enter the search
4. Pangrams: asking for every letter of the alphabet
5. What each one costs

In [1]:
import os
import sys
import time
from collections import Counter

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from crossword import coverage, library
from crossword.fill import Filler
from crossword.index import Index
from crossword.render import show
from crossword.rules import RuleSet
from crossword.words import load

import make_grid

entries = load("crossword/UKACD.txt")
scores = {}
with open("crossword/scores.txt", encoding="utf-8") as handle:
    for line in handle:
        if not line.startswith("#") and line.strip():
            word, value = line.split()
            scores[word] = float(value)
index = Index(entries, scores)
patterns = library.load()

## 1. What a nina is

A message hidden in the grid, read somewhere the solver would not normally
read: around the perimeter, down a diagonal, along an unchecked column. The
answers themselves stay ordinary, and the message only appears to someone who
looks.

The awkward part is that a nina cannot be expressed as a fixed set of cells.
The perimeter of a 15x15 grid is 56 squares, but a grid has blocks on its edges
and the message runs through whatever white squares remain.

In [2]:
perimeter = make_grid.PATHS["perimeter"](15)
print(f"the perimeter of a 15x15 is {len(perimeter)} cells\n")

white = Counter()
for pattern in patterns:
    white[sum(1 for cell in perimeter if cell not in pattern.blocks)] += 1

print("white cells on the perimeter, across the 120 published grids:")
for count, grids in sorted(white.items(), reverse=True)[:6]:
    print(f"  {count:2d} white   {grids:3d} grids")
print(f"\nall 56 open: {white.get(56, 0)} grids")

the perimeter of a 15x15 is 56 cells

white cells on the perimeter, across the 120 published grids:
  56 white     3 grids
  54 white     5 grids
  52 white    34 grids
  50 white    10 grids
  48 white     8 grids
  46 white     2 grids

all 56 open: 3 grids


So the message is laid *relative to the grid*: take the white cells along the
path in order, and let the blocks interrupt it. A setter reads a perimeter nina
by going round and skipping the black squares, and that is exactly how it has
to be placed.

The paths available are the ones a message is worth hiding along.

In [3]:
for name in sorted(make_grid.PATHS):
    cells_on_path = make_grid.PATHS[name](15)
    print(f"  {name:14} {len(cells_on_path):2d} cells")

  antidiagonal   15 cells
  bottomrow      15 cells
  diagonal       15 cells
  leftcol        15 cells
  perimeter      56 cells
  rightcol       15 cells
  toprow         15 cells


## 2. Placing one

`nina_placer` turns a path and a message into a rule the pattern search can
apply: for a given grid, which cell holds which letter. A grid with too few
white cells on the path cannot carry the message and is rejected before any
filling is attempted.

A perimeter nina takes an extra condition. The message must use *every* white
cell on the path, because a circuit that stops three quarters of the way round
is not a perimeter nina. An open path like a diagonal has no such requirement,
and a short message simply sits at the start of it.

In [4]:
path, text = make_grid.read_nina_path("diagonal,IN PLAIN SIGHT")
place = make_grid.nina_placer(path, text)
print(f"message {text.upper()!r} along the diagonal ({len(text)} letters)")

carriers = [p for p in patterns if place(p) is not None]
print(f"{len(carriers)} of {len(patterns)} grids can carry it")

fixed = place(carriers[0])
for cell, letter in list(fixed.items())[:4]:
    print(f"  {cell} <- {letter.upper()}")
print("  ...")

message 'INPLAINSIGHT' along the diagonal (12 letters)
29 of 120 grids can carry it
  (0, 0) <- I
  (1, 1) <- N
  (2, 2) <- P
  (3, 3) <- L
  ...


In [5]:
began = time.time()
result = coverage.best_over_library(
    carriers, index, [], RuleSet(), time_limit=90,
    commonness=3.0, aim=0.85, seed=2, preset=place,
)
print(f"{time.time() - began:.1f}s, complete: {result.ok}")
grid = result.grid
show(grid, highlight=list(fixed))

0.1s, complete: True


## 3. Is it hidden?

A message is only a nina if reading it tells the solver something the answers
do not. If the cells carrying it are exactly a set of whole entries, then
reading them just reads those answers: MYSTERY and THEATRE along a top row
that happens to be two seven-letter entries is not a nina, it is 1 and 5
Across.

So the test is whether any of the message's letters fall outside the entries
that lie wholly inside it.

In [6]:
message_cells = set(make_grid.nina_placer(path, text)(result.pattern))
whole = [s for s in grid.slots() if set(s.cells) <= message_cells]
covered = {c for s in whole for c in s.cells}
exposed = message_cells - covered
touched = {(s.direction, s.row, s.col) for s in grid.slots()
           for c in message_cells if c in s.cells}

print(f"the message spans {len(touched)} entries")
print(f"entries lying wholly inside it: {len(whole)}")
print(f"letters not readable as a whole entry: "
      f"{len(exposed)} of {len(message_cells)}")
print("\nhidden" if exposed else "\nNOT hidden")

reading = "".join(grid.letters[c] for c in path if c in message_cells)
print(f"\nread down the diagonal: {reading.upper()}")

the message spans 12 entries
entries lying wholly inside it: 0
letters not readable as a whole entry: 12 of 12

hidden

read down the diagonal: INPLAINSIGHT


## 4. How fixed letters enter the search

They are simply written into the grid before the search starts, and everything
downstream honours them without being told. A slot's pattern is read from the
grid, so an entry crossing a nina letter already has that letter in its
pattern, and the index will only ever offer words matching it.

The one thing that has to be checked afterwards is whether the message breaks
into real words at the entry boundaries. A message that does not leaves
nonsense round the edge, and it is worth being told which entries those are
rather than having the grid reported as clean.

In [7]:
crossing = [s for s in grid.slots() if set(s.cells) & message_cells]
print("entries the message passes through, and what they became:")
for slot in crossing[:8]:
    letters = sum(1 for c in slot.cells if c in message_cells)
    print(f"  {grid.pattern(slot).upper():16} "
          f"{letters} of its {slot.length} letters fixed by the nina")

entries the message passes through, and what they became:
  TONY             1 of its 4 letters fixed by the nina
  MANTLE           1 of its 6 letters fixed by the nina
  BARBARIAN        1 of its 9 letters fixed by the nina
  UNSWADDLE        1 of its 9 letters fixed by the nina
  EGRETS           1 of its 6 letters fixed by the nina
  OTTO             1 of its 4 letters fixed by the nina
  DIORAMA          1 of its 7 letters fixed by the nina
  CRYPT            1 of its 5 letters fixed by the nina


## 5. Pangrams

A pangram uses every letter of the alphabet at least once; `--pangram 2` asks
for each of them twice.

Nothing is fixed in advance here. Instead the search keeps a mask of the
letters the grid still lacks, and words supplying a missing letter are pulled
forward in the candidate order. The bonus is *per missing letter*, so a word
supplying both a Q and a Z outranks one supplying either — and once a letter is
in the grid its pull vanishes, which is what stops the fill flooding with
awkward words once the rare letters are covered.

In [8]:
def fill_with(pangram, seed=5):
    made = patterns[0].grid()
    worker = Filler(made, index, RuleSet(), seed=seed, commonness=3.0,
                    aim=0.85, pangram=pangram, node_budget=60000)
    began = time.time()
    if not worker.fill():
        return None, None, None
    letters = Counter("".join(made.letters.values()))
    ranks = []
    for slot in made.slots():
        bucket = index.lengths[slot.length]
        word_id = bucket.by_word.get(made.pattern(slot))
        if word_id is not None and bucket.quantile:
            ranks.append(bucket.quantile[word_id])
    return made, letters, (sum(ranks) / len(ranks), time.time() - began)


for want in (0, 1):
    made, letters, stats = fill_with(want)
    missing = [c for c in "abcdefghijklmnopqrstuvwxyz" if letters[c] < 1]
    rare = " ".join(f"{c}:{letters[c]}" for c in "jqxzkvw")
    label = "no requirement" if want == 0 else f"pangram x{want}"
    print(f"{label:16} missing {len(missing):2d} letters "
          f"({''.join(missing).upper() or 'none'})")
    print(f"{'':16} rare letters  {rare}")
    print(f"{'':16} familiarity {stats[0]:.2f}\n")

no requirement   missing  5 letters (JKQXZ)
                 rare letters  j:0 q:0 x:0 z:0 k:0 v:4 w:3
                 familiarity 0.79

pangram x1       missing  0 letters (none)
                 rare letters  j:1 q:1 x:1 z:1 k:2 v:3 w:1
                 familiarity 0.79



In [9]:
import statistics

print("request        filled   familiarity   time")
for want in (0, 1, 2, 3):
    ranks, times, done = [], [], 0
    for seed in range(8):
        made, letters, stats = fill_with(want, seed=seed)
        if made is None:
            continue
        done += 1
        ranks.append(stats[0])
        times.append(stats[1])
    label = "none" if want == 0 else f"pangram x{want}"
    if ranks:
        print(f"  {label:12}    {done}/8       {statistics.mean(ranks):.2f}"
              f"       {statistics.mean(times):5.1f}s")
    else:
        print(f"  {label:12}    0/8       --")

request        filled   familiarity   time
  none            8/8       0.79         0.0s
  pangram x1      8/8       0.79         0.1s
  pangram x2      4/8       0.71         0.2s
  pangram x3      0/8       --


On this grid a single pangram is free: it completes every time and the fill is
no less ordinary than without it. The escalating bonus is why — the pull towards
a rare letter exists only until that letter is in, so a J and a Q are placed
early and the rest of the grid is chosen as it would have been anyway.

Two of each is a different matter. It halves the completions and takes eight
points off the familiarity, because the second J has to go somewhere the first
one did not. Three of each does not come out at all here: it asks for 78 of the
roughly 160 letters in a British grid, which is most of the alphabet's awkward
end several times over.

In [10]:
made, letters, stats = fill_with(1)
show(made)

## 6. What they cost

Both constraints are paid for in the fill, and the currency is familiarity: the
more of the grid a setter dictates, the less freedom the search has to choose
ordinary words for the rest. Over eight seeds on one grid:

A nina costs in a different way. It does not change which words are preferred;
it removes grids from consideration — only the patterns whose path has room for
the message are even tried — and it fixes letters that every crossing entry
must then accommodate.

## Where this ends

That is the whole builder: a dictionary and an index over it, a library of
grids, a rule set per style, a search that seats chosen words and fills the
rest, and these two constraints on top.

- [1. Building a grid](01-building-a-grid.ipynb)
- [2. Words and the index](02-words-and-the-index.ipynb)
- [3. The fill search](03-the-fill-search.ipynb)
- [4. The other styles](04-the-other-styles.ipynb)